# Extending Search Functions

This tutorial shows how to create custom search strategies by extending [`BaseSearch`](https://instadeepai.github.io/alf/api/alf_core/optimizer/search/). We'll implement random and grid search as examples.

## 1. Imports

In [ ]:
import numpy as np
from alf_core.dataclasses import Candidate, LabelledCandidates, Modality, Predictions, State
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig
from alf_core.model.base_model import BaseModel
from alf_core.optimizer.search import BaseSearch
from alf_core.surrogate.surrogate import Surrogate
from alf_core.utils.enums import ProblemType

## 2. Define Custom Search Functions

Implement the `__call__()` method to generate candidate points. The method receives a [`State`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/state/) and must return a list of [`Candidate`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/candidate/) objects with the appropriate [`Modality`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/candidate/).

**Random search** samples points uniformly within bounds. The only required method is `__call__(state, **kwargs)`, returning a list of `Candidate`s.

In [ ]:
class RandomSearch(BaseSearch):
    """Random search in a bounded space."""

    def __init__(self, bounds: tuple[float, float], num_samples: int = 100):
        self.bounds = bounds
        self.num_samples = num_samples

    def __call__(self, state: State, **kwargs) -> list[Candidate]:
        """Generate random candidates within bounds."""
        # Sample uniformly in the search space
        samples = np.random.uniform(low=self.bounds[0], high=self.bounds[1], size=self.num_samples)

        # Convert to Candidate objects
        candidates = [Candidate(data=x, modality=Modality.TABULAR) for x in samples]
        return candidates

**Grid search** instead returns evenly-spaced points. It also overrides the optional `get_metrics()` hook to report search-specific metrics (here, the grid spacing).

In [ ]:
class GridSearch(BaseSearch):
    """Grid search over a discrete set of points."""

    def __init__(self, bounds: tuple[float, float], num_points: int = 50):
        self.bounds = bounds
        self.num_points = num_points

    def __call__(self, state: State, **kwargs) -> list[Candidate]:
        """Generate evenly-spaced grid of candidates."""
        # Create grid points
        grid = np.linspace(self.bounds[0], self.bounds[1], self.num_points)

        # Convert to Candidate objects
        candidates = [Candidate(data=x, modality=Modality.TABULAR) for x in grid]
        return candidates

    def get_metrics(self, state: State) -> dict[str, float]:
        """Optionally return metrics about the search."""
        return {"grid_spacing": (self.bounds[1] - self.bounds[0]) / (self.num_points - 1)}

## 3. Usage Example

Search functions receive the full task `State`. We build a minimal one: an in-memory dataset plus a placeholder surrogate (these strategies ignore the surrogate, but `State` requires one).

In [ ]:
# Search functions receive the full task `State` from the framework. Here we build a
# minimal one: an in-memory dataset (see the "Extending Datasets" tutorial) plus a
# placeholder surrogate. These search strategies don't use the surrogate, but `State`
# requires one.
class InMemoryDataset(BaseDataset):
    """Minimal dataset backed by candidates and labels already in memory."""

    def __init__(self, candidates, labels, config):
        super().__init__(config)
        self._candidates = candidates
        self._labels = labels

    def load_dataset(self) -> LabelledCandidates:
        return LabelledCandidates(candidates=self._candidates, labels=self._labels)


class PlaceholderModel(BaseModel):
    """Unused surrogate model (the search strategies here ignore the surrogate)."""

    def featurise(self, inputs):
        return np.array([c.data for c in inputs]).reshape(-1, 1)

    def train(self, train_data, val_data):
        pass

    def predict(self, candidate_points):
        return Predictions(means=np.zeros(len(candidate_points)))

    def sample(self, condition=None):
        return []

Populate the dataset with dummy tabular data and assemble the `State`.

In [ ]:
# Populate an in-memory dataset with some dummy tabular data
xs = np.linspace(0.0, 10.0, 20)
initial_candidates = [Candidate(data=float(x), modality=Modality.TABULAR) for x in xs]
initial_labels = np.array([float(x) for x in xs])
config = BaseDatasetConfig(
    name="demo",
    modality=Modality.TABULAR,
    seed=0,
    train_ratio=0.6,
    validation_frac=0.2,
    test_ratio=0.2,
    split_type="random",
    problem_type=ProblemType.REGRESSION,
)
dataset = InMemoryDataset(initial_candidates, initial_labels, config)
dataset.setup()

state = State(
    dataset=dataset,
    surrogate=Surrogate(model=PlaceholderModel()),
    round=0,
    acq_batch_size=10,
)

Run **random search**:

In [ ]:
# Random search
random_search = RandomSearch(bounds=(0.0, 10.0), num_samples=50)
random_candidates = random_search(state)
print(f"Random search generated {len(random_candidates)} candidates")
print(f"Sample values: {[c.data for c in random_candidates[:5]]}")

Run **grid search** and inspect its metrics:

In [ ]:
# Grid search
grid_search = GridSearch(bounds=(0.0, 10.0), num_points=20)
grid_candidates = grid_search(state)
print(f"\nGrid search generated {len(grid_candidates)} candidates")
print(f"First 5 values: {[c.data for c in grid_candidates[:5]]}")
print(f"Metrics: {grid_search.get_metrics(state)}")

## Key Points

- **Required method**: `__call__(state, **kwargs)` must return a list of `Candidate` objects
- **State**: Contains current dataset, surrogate, round number, and batch size
- **Search space**: Define bounds, dimensions, or discrete sets based on your problem
- **Return type**: Always return `list[Candidate]` with appropriate modality
- **Optional**: Override `get_metrics()` to return search-specific metrics
- **Integration**: Search results are passed to acquisition functions for scoring
- **Advanced**: Can use `state.dataset` to avoid proposing previously seen points

Common search strategies:
- **Random**: Uniform or adaptive sampling
- **Grid**: Systematic exploration
- **Generator-based**: Use generative model (see `GeneratorSearch` in the framework)
- **Protocol-based**: Custom search protocols (see `ProtocolSearch`)